spring-temperature sensitivity world map (per-range scatterplot insets)

Builds every component of the composite from the pipeline products and assembles the page with
`gsro_analysis.world_maps.plot_temperature_sensitivity_map`. Run top to bottom for a new dataset version;
when the scatter sweep already exists, the setup cell and the composite cell [2] are enough.

```
pipeline products (not built here)         this notebook                                              outputs (figures/<version>/)
──────────────────────────────────         ─────────────                                              ────────────────────────────
mountain-range cube with the ERA5-Land ──► [1] per-range scatterplot sweep ────────────────────────► anomaly_scatterplots/pngs/<stem>_temperature_2m_spring_months_mean.png
  anomaly zonal means (reduce_partials.py)     (spring 2 m temperature anomaly vs onset anomaly)
range metrics table ─────────────────────► choropleth: anomaly_slope (colour) x anomaly_corr (alpha) ─┐
  (range_metrics.py)                                                                                   │
label_layout.csv (curated) ──────────────► which 30 ranges get a scatterplot + anchors (Robinson m) ───┼──► [2] world_maps.plot_temperature_sensitivity_map
data/global_hillshade_robinson.tif ──────► base map (grey 1–231) + generated 60° × 30° graticule ──────┤        └──► global_temperature_sensitivity_map.png
gsro_analysis.colorbars.temperature_sensitivity ► colorbar (YlOrRd reversed, −12 to 0 days per °C) ───┘
```

Everything is versioned by dataset version (`config.version` from `settings.load_config()`): inputs are read
from `aggregated_results/<version>/`, outputs go to `analyses/mountain_ranges/figures/<version>/`. The
regression numbers on the map (slope, r, n) come from `pipeline/scripts/range_metrics.py`; the insets
recompute the same OLS fit per range for display.

In [ ]:
import textwrap

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from gsro_analysis import aggregate, era5, paths, settings, stats, world_maps

config = settings.load_config()          # the dataset version lives in settings.CONFIG_FILE
version = config.version
figdir = paths.figdir('mountain_ranges', version)

# the mountain-range cube in the analyses' view (the analyses' thresholds, runoff_onset_mean_anomaly added) ...
mountains_ds = stats.prepare_mountain_ranges(aggregate.open_aggregate('mountain_ranges', version))
# ... with the per-range ERA5-Land anomaly zonal means (pipeline/scripts/era5_zonal.py, merged by
# reduce_partials) turned into seasonal means: month='spring_months_mean' is what the scatterplots use
climate_vars = [v for v in era5.VARIABLES if v in mountains_ds]
assert climate_vars, "no ERA5 variables in the cube - run pipeline/scripts/era5_zonal.py, then reduce_partials.py"
mountains_ds = xr.merge([mountains_ds.drop_vars(climate_vars), stats.seasonal_means(mountains_ds[climate_vars])])
n_years = len(mountains_ds.water_year)
# per-range metrics (range_metrics.py) joined to the GMBA polygons: anomaly_slope / anomaly_corr = the choropleth
gmba_stats_gdf = stats.range_metrics_gdf(version)   # GMBA polygons + results/<version>/mountain_range_metrics.csv, joined on read
print(f'{version}: {mountains_ds.sizes["mountain_range"]} ranges, {n_years} water years; {len(gmba_stats_gdf)} GMBA rows')

## [1] Per-range scatterplot sweep

One PNG per range (`anomaly_scatterplots/pngs/<stem>_temperature_2m_spring_months_mean.png`, gitignored,
regenerable): the range's mean spring 2 m temperature anomaly (x, ±3.2 °C) against its mean runoff onset
anomaly (y, ±42 days), one point per water year coloured by year (YlGnBu), an OLS line and a box with
n, r and the slope in days per °C. 3.5 × 3 in at 300 dpi, white background, no frame. Also the raw material of
the per-range scatterplot grid in `temperature_sensitivity.ipynb`. `REBUILD_SWEEP = False` skips the sweep when every range already has a PNG in this version's folder.

In [ ]:
# create and save figures of the scatterplots and line of best fit for each mountain range given a list of mountain ranges....

# mountain_ranges = ['Sierra Nevada', 'Klamath Mountains','Great Basin Ranges', 'Colorado Plateau', 'Southern Rocky Mountains',
#                    'Greater Yellowstone Rockies', 'Idaho-Bitterroot Rocky Mountains', 'Columbia Plateau', 'Cascade Range', 
#                    'Central Montana Rocky Mountains','Olympic Mountains', 'Insular Mountains', 'Coast Mountains','British Columbia Interior',
#                    'Columbia Mountains','Canadian Rockies','Far Northern Rockies','Saint Elias Mountains', 'Yukon Intermountain Ranges',
#                    'Mackenzie Mountains','Alaska Intermountain Ranges','Alaska Range','Brooks Range','South-Central Alaska', 'Aleutian Ranges',]

#import seaborn as sns

#sns.set_theme()
scatter_png_dir = paths.figdir('mountain_ranges', config.version, 'anomaly_scatterplots', 'pngs')

mountain_ranges = mountains_ds['mountain_range'].values

# sort mountain ranges by latitude in mountains_ds
mountain_ranges = sorted(mountain_ranges, key=lambda x: mountains_ds.sel(mountain_range=x).centroid_latitude.values)

REBUILD_SWEEP = False   # False: skip when every range already has a PNG here (the composite only needs the files)
if not REBUILD_SWEEP and len(list(scatter_png_dir.glob('*.png'))) >= len(mountain_ranges):
    print(f'{len(mountain_ranges)} scatterplot PNGs already in {scatter_png_dir}; skipping the sweep (REBUILD_SWEEP = False)')
    mountain_ranges = []

var = 'temperature_2m'
month = 'spring_months_mean'

for i,mountain_range in enumerate(mountain_ranges):
    
    if mountain_range not in mountains_ds['mountain_range'].values:
        print(f"{mountain_range} not in dataset, skipping...")
        continue
    else:
        print(f"Processing {i+1}/{len(mountain_ranges)}: {mountain_range}")
    
    f,ax=plt.subplots(figsize=(3.5,3),dpi=300)

    era5_anoms = mountains_ds[var].sel(mountain_range=mountain_range).sel(month=month)#.sel(month=['spring_month_1', 'spring_month_2','spring_month_3']).mean(dim='month')
    melt_anoms = mountains_ds['runoff_onset_mean_anomaly'].sel(mountain_range=mountain_range)
    water_years = era5_anoms['water_year']
    
    
    combined_df = pd.DataFrame({
        'era5_anom': era5_anoms.values,
        'melt_anom': melt_anoms.values,
        'water_year': water_years.values
    }).dropna()
    
    n = len(combined_df)

    # plot temperature anomalies on x axis, melt anomalies on y axis,color should be water year
    plot = ax.scatter(combined_df['era5_anom'],combined_df['melt_anom'],c=combined_df['water_year'],cmap='YlGnBu',vmin=config.water_years[0],vmax=config.water_years[-1], edgecolors='black',s=50,zorder=2)

    # vertical and horizontal dashed lines at 0
    ax.axhline(0, color='black', linestyle=':')
    ax.axvline(0, color='black', linestyle=':')

    # ax.set_xlabel('10-year (WY2015-WY2024) Spring Temperature Anomaly [°C]')
    # ax.set_ylabel('10-year (WY2015-WY2024) Snowmelt Runoff Onset Anomaly [days]')

    ax.set_xlabel(f'Avg. spring 2m temperature anomaly [°C]', fontsize=9)
    ax.set_ylabel('Avg. runoff onset anomaly [days]',fontsize=9)
    
    if var == 'temperature_2m':
        ax.set_xlim([-3.2,3.2])
    ax.set_ylim([-42,42])

    # create a grid
    ax.grid(True, which='both', linestyle='--', linewidth=0.5, zorder=0)

    
    # turn off the frame
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.set_xticks([-3,-2,-1,0,1,2,3])
    ax.set_yticks([-40,-30,-20,-10,0,10,20,30,40])
    
    ax.tick_params(axis='both', 
                   pad=0,
                   length=0, 
                   #labelsize=9
                   )
    
    #change tick label size
    

    # calculate and print correlation coefficient and line of best fit

    try:
        slope, intercept = np.polyfit(combined_df['era5_anom'], combined_df['melt_anom'], 1)

        corr = np.corrcoef(combined_df['era5_anom'], combined_df['melt_anom'])[0, 1]

        title = mountain_range
        if len(title)>40:
            title = "\n".join(textwrap.wrap(title, 40))

        #ax.set_title(f'{title}\nn: {len(combined_df)} | r: {corr:.2f} | slope: {slope:.1f} days/°C')
        ax.set_title('')
        
        stats_text = f'n = {n}\nr = {corr:.2f}\nslope = {slope:.1f} days/°C'
        ax.text(0.97, 0.97, stats_text, 
                transform=ax.transAxes,
                verticalalignment='top',
                horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                #fontsize=9,
                )

        x_smooth = np.linspace(combined_df['era5_anom'].min(), combined_df['era5_anom'].max(), 20)
        expected_melt_anom = slope * x_smooth + intercept
        ax.plot(x_smooth, expected_melt_anom, color='red', linestyle='--',zorder=3)
    

    except Exception as e:
        print(f"Could not calculate fit for {mountain_range}: {e}")
        ax.set_title(f'{mountain_range}\nNo Fit')
        
    safe_filename = mountain_range.replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_').replace('-', '_').replace("*","_")
    safe_filename = safe_filename + f"_{var}_{month}"
    f.savefig(scatter_png_dir / f'{safe_filename}.png', bbox_inches='tight', dpi=300, pad_inches=0.01)

    plt.close(f)

## [3] The composite page: how it is built and every knob

**Page.** A 480 mm wide page in **page millimetres, origin top-left, y down** (`world_maps.Page`; the convention the published layout used, so every position below is a plain number in mm). The project page is 480 × 300 mm with the map item filling it; the default 3 mm top margin cuts the unused ≈ 13 mm above the top row of insets (page ≈ 287 mm). The map item is drawn in World Robinson metres (`ESRI:54030`), extent `world_maps.PAGES['temperature_sensitivity'].extent`, scale 1 : 82.0 M, i.e. `page.m_per_mm` ≈ 82 km per page millimetre. `world_maps.Page.to_page()` / `from_page()` convert between metres and page mm. With `top_margin_mm=3` (default, `world_maps.TOP_MARGIN_MM`) the page top is placed 3 mm above the highest label block and the pictures move with the map; `top_margin_mm=None` keeps the project page and instead shifts a block down when it would poke out.

**Layers, bottom to top.** hillshade (`data/global_hillshade_robinson.tif`, grey stretch 1–231, 0 = outside the ellipse = transparent, read decimated to 4 km pixels) → generated 60° × 30° graticule (white, 50 %, 0.45 pt dashed) → choropleth (`sensitivity_fill`) → callouts → scatterplot images → names → the colorbar picture (white ground, 1 mm frame). The colorbar is its own axes (`gsro_analysis.colorbars` preset, fonts scaled to the picture size).

**A label block** = the scatterplot image with its bottom-left corner 3 pt above the anchor, the range name one text line above the image (centred on it) in the range's own fill colour, and a straight 0.8 mm callout in that colour at 50 % opacity from the centre of the block to the polygon centroid (it passes under the image). Anchors live in `analyses/mountain_ranges/label_layout.csv` in Robinson metres, columns `anom_x`, `anom_y` for this map (the other map has its own pair): **the anchor is the bottom-left corner of the block.** 41 ranges have `display_label = 1`; `show_anom = 0` hides a label on this map only (eleven here, the ranges without a meaningful fit); `display_map = 0` removes a range from the map entirely (four ranges). Insets are the scatterplot PNGs of [1], 46.5 mm wide (the project's `120·90e6/scale` pt rule; height follows the PNG); the filename stem is `world_maps.inset_stem(MapName, Level_04)`.

| I want to… | Do this |
| --- | --- |
| **move a label block** | edit `anom_x`, `anom_y` in `label_layout.csv`. 10 mm to the right = `+10 * page.m_per_mm`; the `placed` table below says where every block landed on the rendered page (`anchor_x_mm`, `anchor_y_mm`, `block_top`, `inset_*`), and `world_maps.PAGES['temperature_sensitivity'].from_page(x_mm, y_mm)` converts a position on the *project* page to metres (on the resized page add the trim printed below to `y_mm`). Say why in the `note` column |
| **add a labelled range** | `display_label = 1`, `show_anom = 1` (and `show_topo` for the other map), both anchor pairs, make sure the inset PNG exists (built in the sweep above), re-run. A range missing from the CSV is drawn but never labelled |
| **hide a label on this map only** | `show_anom = 0` |
| **inset size** | `style=replace(STYLE_A, inset_width_mm=50)`; `None` = the project's rule |
| **name size / weight / colour** | `size_pt` (11 — 12 pt makes "Great Basin Ranges" and "Southern Rocky Mountains" collide with the fixed anchors), `weight` ('bold'), `colored_text` True (the fill colour; `opaque_text` True drops its alpha), font `world_maps.FONT_FAMILIES` (Arial if installed, else Liberation Sans, else DejaVu Sans) |
| **buffer / halo / shadow** | `buffer_mm` 0.4 (`buffer_color` black), `halo_mm` 0.5 (`halo_color` white — a second stroke outside the buffer so dark reds read over the dark ocean), `shadow` False |
| **name-to-inset gap** | `gap_pt` 3 |
| **callouts** | `callout_lw_mm` 0.8, `callout_alpha` 0.502, `callout_origin` 'centroid' (block centre) or 'exterior', `colored_callout` True |
| **the whole label style** | `style=` a `world_maps.LabelStyle`: `STYLE_A` (default), `STYLE_A_QGIS` (published: 10.4 pt, name colour with its alpha, 0.5 mm black buffer, no halo), or `dataclasses.replace(world_maps.STYLE_A, halo_mm=0)` |
| **the published look** | `style=world_maps.STYLE_A_QGIS, top_margin_mm=None` |
| **colorbar position and frame** | `pictures=` a copy of `world_maps.PICTURES['temperature_sensitivity']` with edited `rect=(x, y, w, h)` (project-page mm), `frame_mm`, `background`; the preset artwork keeps its aspect, top-left anchored, fonts scale with it. Wording and ticks: the preset in `gsro_analysis/colorbars.py` |
| **fill ramp / threshold** | `world_maps.sensitivity_fill`: YlOrRd at `clip(-anomaly_slope / 12, 0, 1)` with alpha `interp(anomaly_corr, [-1, 0], [1, 76/255])`, drawn where `display_map = 1` and `anomaly_slope` exists. Keep it in step with the colorbar preset |
| **hillshade / graticule** | `hillshade_decimation=4` (4 km pixels; larger = faster previews), `graticule_deg=(60, 30)` or `None`; `world_maps.HILLSHADE_STRETCH` (1–231) |
| **page, extent, scale** | `world_maps.PAGES` (`Page(width, height, map_height, extent)`; the extent's aspect must equal the map item's; the inset width and (on the sensitivity map) the font size follow the scale) |
| **output** | `out=` (`None` = do not save), `dpi=300`; the colorbar picture rect is `PICTURES['temperature_sensitivity']['colorbar']` |

**Checks.** Every call runs `world_maps.layout_report(fig, placed)` and **warns** when names overlap each other or another range's inset, when anything is off the page, or when the colorbar's ticks and label run into another picture or an inset. The report is in `placed.attrs['layout_report']`; `placed` has one row per drawn inset (page mm) so a collision can be traced to two rows.

**Where the numbers come from.** Page, extent, inset rule, fonts and picture positions reproduce the QGIS project the published temperature-sensitivity world map was laid out in (validated side by side on 2026-09-01), then departed from it for legibility (the `STYLE_A_QGIS` style and `top_margin_mm=None` give the published look). The project, its spec and the transfer notes are archived in the private repo `recreate_global_snowmelt_runoff_onset_analysis_QGIS_figures_in_mpl`.


In [ ]:
fig, placed = world_maps.plot_temperature_sensitivity_map(
    gmba_stats_gdf, version,
    out=figdir / 'global_temperature_sensitivity_map.png',
    # knobs, defaults shown — see the cell above:
    # style=world_maps.STYLE_A, top_margin_mm=world_maps.TOP_MARGIN_MM,
    # pictures=world_maps.PICTURES['temperature_sensitivity'], hillshade_decimation=4, dpi=300,
)
page = placed.attrs['page']
trim = world_maps.PAGES['temperature_sensitivity'].height - page.height
print(f"{len(placed)} scatterplot insets ({int(placed['has_png'].sum())} with a PNG); page {page.width:.0f} x {page.height:.1f} mm "
      f"(project page trimmed by {trim:.1f} mm at the top)")
print({k: v for k, v in placed.attrs['layout_report'].items() if v})   # collisions, if any (also raised as warnings)

Where every block landed (page mm on the rendered page). Use it with the recipes above when moving labels.

In [ ]:
placed.set_index('MapName')[['anchor_x_mm', 'anchor_y_mm', 'inset_w_mm', 'inset_h_mm', 'block_top', 'block_bottom',
                              'callout_x1', 'callout_y1', 'fill_a']].round(2).sort_index()